In [1]:
# Core
import numpy as np
import pandas as pd

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
#StratifiedKFold is cross validation
#gridsearchcv is hyperparameter tuning
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB


# Metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score,recall_score,f1_score,precision_score
from sklearn.metrics import confusion_matrix, classification_report


c:\Users\Dilsh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
#import moels in models folder using joblib
x_train = joblib.load('../Models/x_train.pkl')
y_train = joblib.load('../Models/y_train.pkl')
x_test = joblib.load('../Models/x_test.pkl')
y_test = joblib.load('../Models/y_test.pkl')


In [5]:

models = {
    "Logistic": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),

    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearSVC(dual=False))
    ]),

    "DecisionTree": Pipeline([
        ('model', DecisionTreeClassifier())
    ]),

    "RandomForest": Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ]),

    "LightGBM": Pipeline([
        ('model', LGBMClassifier(random_state=42))
    ]),

    "CatBoost": Pipeline([
        ('model', CatBoostClassifier(verbose=0))
    ]),

    "NaiveBayes": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ])
}

In [6]:
params = {
    "Logistic": {
        'model__C': [0.1, 1, 10]# C is the regularization parameter in logistic regression. It controls the strength of regularization. A smaller C value means stronger regularization, while a larger C value means weaker regularization.
    },

    "SVM": {
        'model__C': [0.1, 1, 10]# C is the regularization parameter in SVM. It controls the trade-off between achieving a low error on the training data and minimizing the norm of the weights. A smaller C value creates a wider margin, while a larger C value creates a narrower margin.
    },

    "DecisionTree": {
        'model__max_depth': [3, 5, 10]# max_depth is the maximum depth of the decision tree. It limits how deep the tree can grow. A smaller max_depth can help prevent overfitting, while a larger max_depth can allow the model to capture more complex patterns in the data.
    },

    "RandomForest": {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 10]
    },

    "LightGBM": {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.1]
    },

    "CatBoost": {
        'model__iterations': [100, 200],
        'model__depth': [4, 6]
    },

    "NaiveBayes": {
        'model__var_smoothing': [1e-9, 1e-8]
    }
}

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#overfiting like kata padan
#underfiting like  can get idea  like

In [8]:

results = []
best_models = {}

for name in models:
    print(f"\n Training {name}...")

    grid = GridSearchCV(
        models[name],
        param_grid=params[name],
        cv=cv,
        scoring='f1',# F1 score is a good metric for imbalanced datasets as it considers both precision and recall.
        n_jobs=-1
    )

    grid.fit(x_train, y_train)

    best_models[name] = grid.best_estimator_

    # Get scores
    if hasattr(grid.best_estimator_, "predict_proba"):
        prob = grid.best_estimator_.predict_proba(x_test)[:, 1]
    else:
        prob = grid.best_estimator_.decision_function(x_test)

    #  Threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.3, 0.7, 0.05):
        y_pred = (prob > t).astype(int)
        f1 = f1_score(y_test, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    results.append({
        "Model": name,
        "Best_F1": best_f1,
        "Threshold": best_thresh,
        "Best_Params": grid.best_params_
    })

    print("Best F1:", best_f1)
    print("Best Threshold:", best_thresh)


 Training Logistic...
Best F1: 0.8711724575598921
Best Threshold: 0.44999999999999996

 Training SVM...
Best F1: 0.8361420843992267
Best Threshold: 0.3

 Training DecisionTree...
Best F1: 0.8700131804515813
Best Threshold: 0.39999999999999997

 Training RandomForest...
Best F1: 0.8724896872454655
Best Threshold: 0.44999999999999996

 Training LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Be

c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
11 fits failed out of a total of 20.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
11 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit

Best F1: 0.8771754143646409
Best Threshold: 0.39999999999999997

 Training NaiveBayes...
Best F1: 0.8588201428808192
Best Threshold: 0.39999999999999997


In [9]:
#save all models
import joblib
for name, model in best_models.items():
    joblib.dump(model, f'../model/{name}_model.pkl')
    